In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import glob
import pandas as pd
import geopandas as gpd
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. Load Core Data
dhs_data = pd.read_csv('/content/DHS_women_violence_module_all_rounds.csv')
manifest = pd.read_csv('/content/MANIFEST(in).csv')

country_mapping = {
    'Benin': 'Benin', 'Burkina Faso': 'Burkina Faso', 'Cote dIvoire': "Cote d'Ivoire",
    'Gambia': 'Gambia', 'Ghana': 'Ghana', 'Liberia': 'Liberia', 'Mali': 'Mali',
    'Mauritania': 'Mauritania', 'Nigeria': 'Nigeria', 'Senegal': 'Senegal',
    'Sierra Leone': 'Sierra Leone', 'Togo': 'Togo', 'Angola': 'Angola',
    'Cameroon': 'Cameroon', 'Chad': 'Chad', 'DR Congo': 'DR Congo',
    'Gabon': 'Gabon', 'Burundi': 'Burundi', 'Comoros': 'Comoros', 'Ethiopia': 'Ethiopia',
    'Kenya': 'Kenya', 'Madagascar': 'Madagascar', 'Malawi': 'Malawi',
    'Mozambique': 'Mozambique', 'Rwanda': 'Rwanda', 'Tanzania': 'Tanzania',
    'Uganda': 'Uganda', 'Zambia': 'Zambia', 'Zimbabwe': 'Zimbabwe', 'Lesotho': 'Lesotho',
    'Namibia': 'Namibia', 'South Africa': 'South Africa'
}

# Ensure output directory exists
output_dir = '/content/Processed_Country_Data'
os.makedirs(output_dir, exist_ok=True)

# Helper function to find a shapefile based on survey ID
def find_shapefile(survey_id, base_path='/content/drive/MyDrive'):
    # Search for any .shp file that has the survey_id in its path
    search_pattern = os.path.join(base_path, f'**/*{survey_id}*/**/*.shp')
    matches = glob.glob(search_pattern, recursive=True)
    return matches[0] if matches else None

for dict_name, manifest_name in country_mapping.items():
    print(f"\n================ Processing: {dict_name} ================")

    # 1. Find surveys for this country in the manifest (UPDATED to 'country')
    country_surveys = manifest[manifest['country'] == manifest_name]['survey'].unique().tolist()
    if not country_surveys:
        print(f"No surveys found in manifest for {manifest_name}. Skipping.")
        continue

    country_dhs = dhs_data[dhs_data['survey'].isin(country_surveys)].copy()

    merged_years = []
    for survey_id in country_surveys:
        year_dhs = country_dhs[country_dhs['survey'] == survey_id].copy()
        if len(year_dhs) == 0: continue

        shp_path = find_shapefile(survey_id)
        if not shp_path:
            print(f"  Missing Shapefile for {survey_id}. Skipping this round.")
            continue

        print(f"  Merging {survey_id}...")
        try:
            year_geo = gpd.read_file(shp_path)
            year_geo = year_geo[year_geo['LATNUM'] != 0.0].copy()

            year_geo['DHSCLUST'] = year_geo['DHSCLUST'].astype(float).astype(int)
            year_dhs['v001'] = year_dhs['v001'].astype(float).astype(int)

            year_merged = year_geo.merge(year_dhs, left_on='DHSCLUST', right_on='v001', how='inner')
            merged_years.append(year_merged)
        except Exception as e:
            print(f"  Error merging {survey_id}: {e}")

    if not merged_years:
        print(f"Failed to merge any data for {dict_name}.")
        continue

    final_country_data = pd.concat(merged_years, ignore_index=True)

    # 2. Date Calculation & Data Prep
    int_year = ((final_country_data['v008'] - 1) // 12) + 1900
    int_month = final_country_data['v008'] - ((int_year - 1900) * 12)
    final_country_data['interview_date'] = pd.to_datetime(
        int_year.astype(int).astype(str) + '-' + int_month.astype(int).astype(str).str.zfill(2) + '-15',
        format='%Y-%m-%d'
    )

    final_country_data['v000'] = final_country_data['survey']
    final_country_data['informal'] = ((final_country_data['sought_any'] == 1) & (final_country_data['formal'] == 0)).astype(int)

    # Restrict to recent victims
    final_country_data = final_country_data[final_country_data['victim_12m'] == 1].copy()

    # Standardize region column for FE clustering
    final_country_data['region_name'] = final_country_data['v024'].astype(str)

    out_path = os.path.join(output_dir, f"{dict_name}_DHS_Spatial_Merged.csv")
    final_country_data.to_csv(out_path, index=False)
    print(f"Saved {len(final_country_data)} victim records for {dict_name}.")